[目录](./table_of_contents.ipynb)

# 粒子滤波器

In [ ]:
%matplotlib inline

In [ ]:
#格式化书籍
import book_format
book_format.set_style()

## 动机

这是我们的问题。我们有移动的物体需要追踪。它们可能是战斗机和导弹，也可能是我们追踪在田野上玩板球的人。都无所谓。我们学习了哪些滤波器能够处理这个问题呢？不幸的是，它们中没有一个是理想的。让我们思考一下这个问题的特征。

* **多峰性(multimodal)**: 我们想同时追踪零、一个或多个对象。

* **遮挡(occlusions)**: 一个对象可能会隐藏另一个对象，导致一个测量值对应多个对象。

* **非线性行为(nonlinear behavior)**: 飞机受到风的摆动，球呈抛物线运动，人们相互碰撞。

* **非线性测量**：雷达可以给我们物体的距离。将其转换为（x，y，z）坐标需要进行平方根运算，这是非线性的。

* **非高斯噪声**：当物体穿过背景时，计算机视觉可能会将背景的一部分误认为是物体。

* **连续性**：物体的位置和速度（即状态空间）可以随时间平滑变化。

* **多元化**：我们想要跟踪几个属性，例如位置、速度、转向速率等。

* **未知过程模型**：我们可能不知道系统的过程模型。

我们学习的滤波器都无法很好地处理所有这些约束条件。

* **离散贝叶斯滤波器**：这种滤波器具有大部分属性。它是多峰的，可以处理非线性测量，并可以扩展到处理非线性行为。但它是离散的和单变量的。

* **卡尔曼滤波器**：卡尔曼滤波器可为具有高斯噪声的单峰线性系统生成最佳估计值。 然而，我们的问题都不满足这些条件。

* **无迹卡尔曼滤波器**：UKF处理非线性，连续，多元问题。然而，它不是多峰的，也不能处理遮挡。它可以处理适度非高斯噪声，但不适用于非常非高斯分布或非常非线性的问题。

* **扩展卡尔曼滤波器**：EKF与UKF具有相同的优点和局限性，只是更加敏感于强非线性和非高斯噪声。

## 蒙特卡罗采样

在UKF章节中，我生成了一个类似于此图的图表以说明非线性系统对高斯分布的影响：

In [ ]:
import kf_book.pf_internal as pf_internal
pf_internal.plot_monte_carlo_ukf()

左图显示了基于高斯分布的3,000个点。

$$\mu = \begin{bmatrix}0\\0\end{bmatrix},\, \, \, \Sigma = \begin{bmatrix}32&15\\15&40\end{bmatrix}$$

右图显示这些点通过以下一组方程式：

$$\begin{aligned}x&=x+y\\
y &= 0.1x^2 + y^2\end{aligned}$$ 

使用有限数量的随机采样点计算结果称为[*蒙特卡罗*](https://en.wikipedia.org/wiki/Monte_Carlo_method)（MC）方法。思想很简单。生成足够的点以获取问题的代表样本，将点通过您正在建模的系统运行，然后计算转换点上的结果。

简而言之，这就是粒子滤波器所做的。我们将贝叶斯滤波算法应用于成千上万的粒子，其中每个粒子代表系统的一个*可能*状态。我们使用粒子的加权统计量从成千上万的粒子中提取估计状态。

## 通用粒子滤波器算法

1. **随机生成一堆粒子**

In [ ]:
粒子可以有位置、航向和/或其他需要估计的状态变量。每个粒子都有一个权重（概率），表示它匹配系统实际状态的可能性。用相同的权重初始化每个粒子。

2. **预测粒子的下一个状态**

In [ ]:
基于您对真实系统行为的预测移动粒子。

3. **更新**

In [ ]:
根据测量值更新粒子的权重。与测量值匹配较好的粒子比不太匹配的粒子具有更高的权重。

4. **重采样**

In [ ]:
丢弃高度不可能的粒子，并用更可能的粒子的副本替换它们。

5. **计算估计值**

In [ ]:
可选地，计算粒子集的加权平均值和协方差以获得状态估计值。

这种天真的算法存在实际困难，我们需要克服，但这是一般的想法。让我们看一个例子。我为来自UKF和EKF章节的机器人定位问题编写了一个粒子滤波器。机器人具有转向和速度控制输入。它有测量到可见地标的距离的传感器。传感器和控制机制都有噪声，我们需要跟踪机器人的位置。

在这里，我运行了一个粒子滤波器并绘制了粒子的位置。左侧的图是一次迭代后的图像，右侧的图是十次迭代后的图像。红色的“X”显示机器人的实际位置，大圆圈是计算的加权平均位置。

In [ ]:
pf_internal.show_two_pf_plots()

如果您在浏览器中查看此内容，则此动画显示整个序列：

<img src='animations/particle_filter_anim.gif'>

第一次迭代后，粒子仍然随机散布在地图周围，但是你可以看到一些粒子已经聚集在机器人的位置附近。计算出的平均值非常接近机器人的位置。这是因为每个粒子的权重是基于其与测量值的相似程度来确定的。机器人附近(1,1)，所以接近(1,1)的粒子将具有很高的权重，因为它们与测量值非常接近。远离机器人的粒子将不匹配测量值，因此其权重非常低。估计位置是粒子位置的加权平均值。靠近机器人的粒子对计算的贡献更大，因此估计值相当准确。

几次迭代之后，你会发现所有的粒子都聚集在机器人周围。这是由于“重采样”步骤。重采样会丢弃非常不可能的粒子(权重非常低)并用概率更高的粒子替换它们。

我还没有完全说明为什么这个方法有效，也没有完全解释粒子加权和重采样算法，但它应该很容易理解。先生成一堆随机粒子，让它们“有点”跟随机器人移动，根据它们与测量值匹配的程度为它们加权，只保留可能性大的粒子。这样看来，它应该有效，而且的确有效。

## 通过蒙特卡罗进行概率分布

假设我们想要知道函数 $y= \mathrm{e}^{\sin(x)}$ 在区间 [0, $\pi$] 下方的面积。该面积可以通过定积分 $\int_0^\pi  \mathrm{e}^{\sin(x)}\, \mathrm{d}x$ 来计算。现在，请尝试计算答案；我等着你。

如果你很聪明的话就不要接受这个挑战；$\mathrm{e}^{\sin(x)}$ 无法被解析地积分。世界上有很多我们无法积分的方程。例如，考虑计算物体的亮度。一些光线会被物体反射。一些反射光线会反弹到其他物体上并重新照射到原始物体上，增加亮度。这就产生了一个*递归积分*。祝你好运。

然而，使用蒙特卡罗技术计算积分非常简单。要找到曲线下的面积，请创建一个包含所需区间内曲线的边界框。在框内生成随机位置的点，并计算落在曲线下的点与总点数的比率。例如，如果40%的点在曲线下方，而边界框的面积为1，则曲线下的面积近似为0.4。随着点数趋近于无限，您可以实现任意精度。在实践中，几千个点将为您提供相当准确的结果。

您可以使用此技术数值积分任意难度的函数。这包括不可积和不连续的函数。这项技术是由斯坦利·乌拉姆在洛斯阿拉莫斯国家实验室发明的，以使他能够进行无法通过纸质计算解决的核反应计算。

让我们通过计算圆的面积来计算$\pi$。我们将定义一个半径为1的圆，并将其限制在一个正方形中。正方形的边长为2，因此面积为4。我们在框内生成一组均匀分布的随机点，并计算有多少个落在圆内。圆的面积被计算为盒子的面积乘以圆内点数与总点数的比率。最后，我们知道$A = \pi r^2$，所以我们计算$\pi = A / r^2$。

我们首先创建点。

In [ ]:
N = 20000
pts = uniform(-1, 1, (N, 2))

如果一个点到圆心的距离小于或等于半径，则该点在圆内。我们使用`numpy.linalg.norm`计算距离，它计算向量的大小。由于向量始于(0, 0)，因此调用norm将计算点到原点的距离。

In [ ]:
dist = np.linalg.norm(pts, axis=1)

接下来，我们计算符合条件的距离。以下代码将返回一个布尔值数组，其中包含符合条件 `dist <= 1` 的值为 `True`：

In [ ]:
in_circle = dist <= 1

接下来要做的就是计算圆内的点数，计算 π，并绘制结果。我把所有代码都放在一个单元格中，这样您就可以尝试使用不同的 `N` 值，即点数。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numpy.random import uniform 

N = 20000  # 点数
radius = 1.
area = (2*radius)**2

pts = uniform(-1, 1, (N, 2))

# 距离 (0,0) 的距离
dist = np.linalg.norm(pts, axis=1)
in_circle = dist <= 1

pts_in_circle = np.count_nonzero(in_circle)
pi = 4 * (pts_in_circle / N)

# 绘制结果
plt.scatter(pts[in_circle,0], pts[in_circle,1], 
            marker=',', edgecolor='k', s=1)
plt.scatter(pts[~in_circle,0], pts[~in_circle,1], 
            marker=',', edgecolor='r', s=1)
plt.axis('equal')

In [ ]:
print(f'pi的平均值(N={N})= {pi:.4f}')
print(f'pi的误差(N={N})= {np.pi-pi:.4f}')

这个洞察力引导我们认识到，我们可以使用 Monte Carlo 方法来计算任何概率分布的概率密度。例如，假设我们有这个高斯分布：

In [ ]:
from filterpy.stats import plot_gaussian_pdf
plot_gaussian_pdf(mean=2, variance=3);

概率密度函数（PDF）给出了随机变量落在 2 个值之间的概率。例如，我们可能想知道上图中 x 落在 0 和 2 之间的概率。这是一个连续函数，因此我们需要对其进行积分才能找到曲线下的面积，因为该面积等于该值范围内发生的概率。 

$$P[a \le X \le b] = \int_a^b f_X(x) \, dx$$

对于高斯函数来说，计算这个积分是很容易的。但现实生活并不那么简单。例如，下面的图表显示了一个概率分布。没有办法对任意曲线进行解析描述，更不用说积分了。

In [ ]:
pf_internal.plot_random_pd()

我们可以使用蒙特卡罗方法来计算任何积分。概率密度函数是通过积分计算的，因此我们可以使用蒙特卡罗方法计算该曲线的概率密度函数。

## 粒子滤波器

所有这些都引导我们来到粒子滤波器。考虑在城市环境中跟踪机器人或汽车。为了保持一致性，我将使用 EKF 和 UKF 章节中的机器人定位问题。在这个问题中，我们跟踪一个机器人，该机器人具有测量到已知地标的距离和方位角的传感器。

粒子滤波器是一类算法。我介绍了一种特定形式的粒子滤波器，它易于理解，并与我们在本书中研究的问题相关。这将使一些步骤看起来有些“神奇”，因为我没有提供完整的解释。这将在本章后面进行解释。

参考上一节的讨论，我们首先创建了数千个“粒子”。每个粒子都有一个位置，表示机器人在场景中可能存在的位置，以及可能的方向和速度。假设我们没有关于机器人位置的任何知识。我们想要在整个场景中均匀地分散这些粒子。如果您将所有粒子视为概率分布，则有更多粒子的位置表示更高的置信度，而粒子较少的位置则表示置信度较低。如果在特定位置附近有大量粒子的集合，那将意味着我们更加确定机器人在那里。

每个粒子需要一个权重——理想情况下它代表机器人真实位置的概率。这个概率很难计算，因此我们只需要让它与该概率成*比例*，这是可计算的。在初始化时，我们没有理由支持一个粒子胜过另一个粒子，因此我们为$N$个粒子分配了$1/N$的权重。我们使用$1/N$是为了使所有概率的总和等于1。

粒子和权重的组合形成了我们问题的*概率分布*。回想一下*离散贝叶斯*一章。在那一章中，我们将走廊中的位置建模为离散和均匀分布的。这非常类似，只是粒子在连续空间中随机分布，而不是被限制在离散位置上。在这个问题中，机器人可以在一个平面上移动，该平面的右下角位于(0,0)。

为了追踪我们的机器人，我们需要维护x，y和heading的状态。我们将在一个（N，3）形状的数组中存储`N`个粒子。三列依次包含x，y和heading。

如果您正在被动地跟踪某物（没有控制输入），则需要在状态中包含速度并使用该估计来进行预测。更多维度需要指数级数量的粒子才能形成良好的估计，因此我们总是试图最小化状态中的随机变量数量。

此代码在区域内创建均匀和高斯分布的粒子：

In [ ]:
from numpy.random import uniform

def create_uniform_particles(x_range, y_range, hdg_range, N):
    particles = np.empty((N, 3))
    particles[:, 0] = uniform(x_range[0], x_range[1], size=N)
    particles[:, 1] = uniform(y_range[0], y_range[1], size=N)
    particles[:, 2] = uniform(hdg_range[0], hdg_range[1], size=N)
    particles[:, 2] %= 2 * np.pi
    return particles

In [ ]:
def create_gaussian_particles(mean, std, N):
    particles = np.empty((N, 3))
    particles[:, 0] = mean[0] + (randn(N) * std[0])
    particles[:, 1] = mean[1] + (randn(N) * std[1])
    particles[:, 2] = mean[2] + (randn(N) * std[2])
    particles[:, 2] %= 2 * np.pi
    return particles

例如：

In [ ]:
create_uniform_particles((0,1), (0,1), (0, np.pi*2), 4)

### 预测步骤

Bayes 算法中的预测步骤使用过程模型来更新对系统状态的信念。那么，如果我们使用粒子，该如何实现呢？每个粒子代表机器人的一个可能位置。假设我们向机器人发送一个命令，让它向前移动 0.1 米，并旋转 0.007 弧度。我们可以将每个粒子移动这个距离。但这样很快就会遇到问题。机器人的控制并不完美，因此它不会像命令一样精确地移动。因此，我们需要给粒子的运动添加噪声，以有合理的机会捕捉机器人实际的运动。如果您不对系统的不确定性建模，粒子滤波器将无法正确地模拟我们对机器人位置的信念的概率分布。

In [ ]:
def predict(particles, u, std, dt=1.):
    """根据控制输入 u（偏航变化，速度）和noise Q（标准偏航变化，标准速度）移动"""

N = len(particles)
    # 更新航向
    particles[:, 2] += u[0] + (randn(N) * std[0])
    particles[:, 2] %= 2 * np.pi

In [ ]:
# 按照（有噪声的）指令方向移动
dist = (u[1] * dt) + (randn(N) * std[1])
particles[:, 0] += np.cos(particles[:, 2]) * dist
particles[:, 1] += np.sin(particles[:, 2]) * dist

In [ ]:

### 更新步骤

接下来，我们会得到一组测量值——每个当前可见地标的一个。我们应该如何使用这些测量值来改变由粒子模型建模的概率分布？

回想一下 **离散贝叶斯** 章节。在那一章中，我们将走廊中的位置建模为离散和均匀分布的。我们为每个位置分配了一个我们称之为 *prior* 的概率。当出现新的测量值时，我们将该位置的当前概率（*prior*）乘以该测量值匹配该位置的 *likelihood*：

```python
def update(likelihood, prior):
    posterior = prior * likelihood
    return normalize(posterior)

这是一个实现方程的代码

$$x = \| \mathcal L \bar x \|$$

这是贝叶斯定理的体现：

$$\begin{aligned}P(x \mid z) &= \frac{P(z \mid x)\, P(x)}{P(z)} \\
 &= \frac{\mathtt{似然}\times \mathtt{先验}}{\mathtt{标准化}}\end{aligned}$$

我们对粒子执行相同的操作。每个粒子都有一个位置和权重，用于估计其与测量的匹配程度。将权重标准化，使它们总和为1，就可以将其转化为概率分布。距离机器人最近的粒子通常比离机器人远的粒子具有更高的权重。

In [ ]:
def update(particles, weights, z, R, landmarks):
    for i, landmark in enumerate(landmarks):
        distance = np.linalg.norm(particles[:, 0:2] - landmark, axis=1)
        weights *= scipy.stats.norm(distance, R).pdf(z[i])

weights += 1.e-300      # 避免舍入为零
    weights /= sum(weights) # 正则化

在文献中，该算法的这一部分被称为*顺序重要采样*（Sequential Importance Sampling，SIS）。权重方程被称为*重要密度*（importance density）。我将在下一节中介绍这些理论基础。但现在我希望这个直觉上有意义。如果我们根据它们与测量的匹配程度对粒子进行加权，它们可能是合适的样本，用于合并测量后系统的概率分布。理论证明了这一点。权重是贝叶斯定理中的*似然度*（likelihood）。不同的问题需要以稍微不同的方式解决这一步骤，但这是一般的想法。

### 计算状态估计

在大多数应用程序中，您将希望在每次更新之后了解估计状态，但滤波器仅由一组粒子组成。假设我们正在跟踪一个对象（即它是单峰的），我们可以将估计的平均值计算为粒子的加权值之和。

$$\displaystyle \mu = \frac{1}{N}\sum_{i=1}^N w^ix^i$$

这里我采用符号 $x^i$ 表示第 $\mathtt{i}$ 个粒子。使用上标是因为我们经常需要使用下标来表示时间步，例如 $\mathtt{k+1}$ 步骤会变得笨拙，即 $x^i_{k+1}$。

这个函数计算粒子的平均值和方差:

In [ ]:
def estimate(particles, weights):
    """返回加权粒子的均值和方差"""

    pos = particles[:, 0:2]
    mean = np.average(pos, weights=weights, axis=0)
    var  = np.average((pos - mean)**2, weights=weights, axis=0)
    return mean, var

如果我们在一个大小为1x1的正方形中创建具有相等权重的点的均匀分布，我们会得到一个非常接近正方形中心（0.5, 0.5）的平均位置和一个小方差。

In [ ]:
particles = create_uniform_particles((0,1), (0,1), (0, 5), 1000)
weights = np.array([.25]*1000)
estimate(particles, weights)

### 粒子重采样

SIS算法受到*退化问题*的影响。它从具有相等权重的均匀分布的粒子开始。可能只有少数粒子靠近机器人。随着算法运行，任何不符合测量的粒子都将获得极低的权重。只有靠近机器人的粒子才会有明显的权重。我们可能有5,000个粒子，但只有3个对状态估计有实质性的贡献！我们称过滤器已经*退化*。这个问题通常通过对粒子进行某种形式的*重采样*来解决。

非常轻的粒子不能有效地描述机器人的概率分布。重新采样算法会丢弃概率非常低的粒子，并用具有较高概率的新粒子替换它们。它通过复制相对概率较高的粒子来完成这一过程。副本通过预测步骤中添加的噪声略微分散。这样就得到了一组点，其中大多数粒子准确地表示了概率分布。

有许多重新采样算法。现在让我们来看看其中一种最简单的——*简单随机重采样*，也称为*多项式重采样*。它从当前粒子集合中采样$N$次，从样本中生成新的粒子集合。选择任何给定粒子的概率应与其权重成比例。

我们使用NumPy的`cumsum`函数来完成这个操作。`cumsum`函数计算数组的累计和。也就是说，第一个元素是第零个和第一个元素的和，第二个元素是第零个、第一个和第二个元素的和，以此类推。然后我们在0.0到1.0的范围内生成随机数，并进行二分查找以找到最接近该数字的权重：

In [ ]:
def simple_resample(particles, weights):
    N = len(particles)
    cumulative_sum = np.cumsum(weights)
    cumulative_sum[-1] = 1. # 避免舍入误差
    indexes = np.searchsorted(cumulative_sum, random(N))

    # 根据索引进行重采样
    particles[:] = particles[indexes]
    weights.fill(1.0 / N)

我们不会在每个时期重新采样。例如，如果您没有收到新的测量数据，则没有任何信息可以从中受益。我们可以使用称为*有效 N*的东西来确定何时重新采样，它大致测量有意义地对概率分布做出贡献的粒子数量。其方程为

$$\hat{N}_\text{eff} = \frac{1}{\sum w^2}$$

我们可以使用以下 Python 代码实现：

In [ ]:
def neff(weights):
    return 1. / np.sum(np.square(weights))

如果 $\hat{N}_\text{eff}$ 低于某个阈值，就该重新采样了。一个有用的起点是 $N/2$，但这取决于具体问题。 $\hat{N}_\text{eff} = N$ 也是可能的，这意味着粒子集合已经坍缩成为一个点（每个粒子的权重相等）。虽然这可能不是理论上的纯粹，但如果发生这种情况，我会创建一个新的粒子分布，希望能够产生更多样化的粒子。如果这种情况经常发生，你可能需要增加粒子数，或者调整滤波器。我们稍后会更详细地讨论这个问题。

## SIR Filter - 完整示例

还有更多需要学习的，但我们已经知道足够实现完整的粒子滤波器了。我们将实现*采样重要性重采样滤波器*，即 SIR。

我需要介绍一种比上面更复杂的重采样方法。FilterPy提供了几种重采样方法，稍后我会描述它们。它们接收一个权重数组，并返回已被选中进行重采样的粒子的索引。我们只需要编写一个函数，从这些索引执行重采样：

In [ ]:
def resample_from_index(particles, weights, indexes):
    particles[:] = particles[indexes]
    weights.resize(len(particles))
    weights.fill (1.0 / len(weights))

为了实现滤波器，我们需要创建粒子和地标。然后，我们执行一个循环，依次调用`predict`，`update`，重采样，然后使用`estimate`计算新的状态估计值。

In [ ]:
from filterpy.monte_carlo import systematic_resample
from numpy.linalg import norm
from numpy.random import randn
import scipy.stats

In [ ]:
def run_pf1(N, iters=18, sensor_std_err=.1,
            do_plot=True, plot_particles=False,
            xlim=(0, 20), ylim=(0, 20),
            initial_x=None):
    landmarks = np.array([[-1, 2], [5, 10], [12,14], [18,21]])
    NL = len(landmarks)
    
    plt.figure()
   
    # 创建粒子和权重
    if initial_x is not None:
        particles = create_gaussian_particles(
            mean=initial_x, std=(5, 5, np.pi/4), N=N)
    else:
        particles = create_uniform_particles((0,20), (0,20), (0, 6.28), N)
    weights = np.ones(N) / N

    if plot_particles:
        alpha = .20
        if N > 5000:
            alpha *= np.sqrt(5000)/np.sqrt(N)           
        plt.scatter(particles[:, 0], particles[:, 1],
                    alpha=alpha, color='g')
    
    xs = []
    robot_pos = np.array([0., 0.])
    for x in range(iters):
        robot_pos += (1, 1)

# 机器人到每个地标的距离
        zs = (norm(landmarks - robot_pos, axis=1) + 
              (randn(NL) * sensor_std_err))

In [ ]:
# 沿着对角线向前移动到 (x+1, x+1)
predict(particles, u=(0.00, 1.414), std=(.2, .05))

# 合并测量结果
update(particles, weights, z=zs, R=sensor_std_err, 
       landmarks=landmarks)

# 如果有效粒子太少，则重新采样
if neff(weights) < N/2:
    indexes = systematic_resample(weights)
    resample_from_index(particles, weights, indexes)
    assert np.allclose(weights, 1/N)
mu, var = estimate(particles, weights)
xs.append(mu)

如果需要绘制粒子图：

In [ ]:
plt.scatter(particles[:, 0], particles[:, 1], 
                        color='k', marker=',', s=1)
p1 = plt.scatter(robot_pos[0], robot_pos[1], marker='+',
                         color='k', s=180, lw=3)
p2 = plt.scatter(mu[0], mu[1], marker='s', color='r')

将xs转换为numpy数组：

In [ ]:
xs = np.array(xs)

绘制散点图：

In [ ]:
plt.legend([p1, p2], ['实际值', 'PF'], loc=4, numpoints=1)
plt.xlim(*xlim)
plt.ylim(*ylim)
plt.show()

打印最终位置误差和方差：

In [ ]:
print('final position error, variance:\n\t', mu - np.array([iters, iters]), var)

随机种子为2：

In [ ]:
from numpy.random import seed
seed(2) 

执行PF1：

In [ ]:
run_pf1(N=5000, plot_particles=False)

大部分代码用于初始化和绘图。粒子滤波器的全部处理都包含在以下代码中：

In [ ]:
# 向对角线前进到(x+1, x+1)
predict(particles, u=(0.00, 1.414), std=(.2, .05))

# 合并测量结果
update(particles, weights, z=zs, R=sensor_std_err, 
       landmarks=landmarks)
       
# 如果有效粒子太少则重新采样
if neff(weights) < N/2:
    indexes = systematic_resample(weights)
    resample_from_index(particles, weights, indexes)

mu, var = estimate(particles, weights)

In [ ]:

第一行预测了粒子的位置，假设机器人直线运动 (`u[0] == 0`)，在 x 和 y 轴上移动了 1 个单位 (`u[1] == 1.414`)。转向误差的标准差为 0.2，距离的标准差为 0.05。当此调用返回时，粒子都已向前移动，但权重不再正确，因为它们尚未更新。

下一行将测量结果纳入滤波器中。这并不改变粒子的位置，只改变权重。如果您还记得，粒子的权重是根据它与传感器误差模型的gaussian相匹配的概率计算的。距离测量距离越远的粒子，它是好的表示的可能性就越小。

最后两行示例有效粒子数($\hat{N}_\text{eff}$)。如果它低于 $N/2$，我们进行重采样，以确保我们的粒子形成了实际概率分布的良好表示。

现在让我们看看所有粒子的绘制情况。交互式地观察这个过程会更具有指导意义，但这种格式仍然为我们提供了有用的信息。我用非常淡的绿色和大圆圈绘制了原始随机分布的点，以帮助将它们与随后的迭代区分开来，其中粒子用黑色像素绘制。粒子数量很大，所以很难看到细节，因此我将迭代次数限制为8次，以便我们可以放大并仔细观察。

```python
seed(2)
run_pf1(N=5000, iters=8, plot_particles=True, 
        xlim=(0,8), ylim=(0,8))

从图中看起来，在前两个机器人位置只有很少的粒子。但事实并非如此；有5,000个粒子，但由于重采样，大多数是重复的。原因是传感器的高斯函数非常窄。这被称为*样本贫化*，可能导致滤波器发散。我将在下面详细介绍这个问题。现在，在观察x=2的第二步时，我们可以看到粒子已经分散了一点。这种分散是由于运动模型的噪声造成的。所有粒子根据控制输入“u”向前投影，但是对于每个粒子，会加入与机器人控制机制误差成比例的噪声。到第三步，粒子已经分散得足够多，可以形成一个令人信服的粒子云围绕机器人。

粒子云的形状是椭圆形，这不是巧合。传感器和机器人控制都被建模为高斯分布，因此系统的概率分布也是高斯分布。粒子滤波器是概率分布的采样，所以粒子云应该是一个椭圆形。

需要注意的是，粒子滤波算法*不需要*传感器或系统是高斯分布或线性的。因为我们用粒子云来表示概率分布，所以我们可以处理任何概率分布和非常非线性的问题。概率模型中可能存在不连续和硬限制。

### 传感器误差对滤波器的影响

滤波器的前几次迭代导致了许多重复的粒子。这是因为传感器的模型是高斯的，并且我们给了它一个小的标准差，即 $\sigma=0.1$。这在一开始是不直观的。当噪声较小时，卡尔曼滤波器的表现更好，但是粒子滤波器的表现可能更差。

我们可以推理出为什么会出现这种情况。如果 $\sigma=0.1$，机器人在 (1, 1) 处，而一个粒子在 (2, 2) 处，那么这个粒子距离机器人相差 14 个标准差。这使得它的概率接近零。它对均值的估计没有任何贡献，并且在重采样之后极不可能存活。如果 $\sigma=1.4$，那么这个粒子只有 $1\sigma$ 的距离，因此它将对均值的估计有所贡献。在重采样期间，它很可能会被复制一次或多次。

这一点*非常重要* - 一个非常准确的传感器可能导致滤波器性能不佳，因为很少有粒子是概率分布的良好样本。我们有几个可用的解决方案。首先，我们可以人为地增加传感器噪声标准差，以便粒子滤波器将更多点视为与机器人的概率分布相匹配。这是非最优的，因为其中一些点将不匹配。真正的问题在于没有足够多的点生成，使得足够多的点靠近机器人。增加“N”通常可以解决这个问题。这个决定不是成本免费的，因为增加粒子数量显著增加了计算时间。不过，让我们看看使用100,000个粒子的结果。

In [ ]:
seed(2) 
run_pf1(N=100000, iters=8, plot_particles=True, 
        xlim=(0,8), ylim=(0,8))

在 x=1 处有更多粒子，并且在 x=2 处有一个令人信服的云。 显然，滤波器的性能更好，但代价是大量的内存使用和长时间的运行时间。

另一种方法是更加聪明地生成初始粒子云。 假设我们猜测机器人在 (0, 0) 附近。 这并不精确，因为模拟实际上将机器人放置在 (1, 1) 处，但它很接近。 如果我们在 (0, 0) 附近创建一个正态分布云，那么粒子与机器人的位置匹配的可能性就更大。

`run_pf1()` 有一个可选参数 `initial_x`。 使用它来指定机器人的初始位置猜测。 然后，代码使用 `create_gaussian_particles(mean, std, N)` 来创建围绕初始猜测正常分布的粒子。 我们将在下一节中使用它。

### 样本不足导致滤波器退化

目前的滤波器远非完美。 这是它在不同的随机种子下的性能。

In [ ]:
seed(6) 
run_pf1(N=5000, plot_particles=True, ylim=(-20, 20))

这里的初始样本点没有生成任何接近机器人的点。粒子滤波器在重采样操作期间不会创建新点，因此会重复点，这些点不代表概率分布的样本。正如前面提到的那样，这被称为*样本贫化*。问题很快就失控了。粒子与地形测量不匹配，因此它们在高度非线性的曲线分布中变得分散，粒子滤波器与现实发生偏差。机器人附近没有可用的粒子，因此它永远无法收敛。

让我们利用`create_gaussian_particles()`方法尝试在机器人附近生成更多的点。我们可以使用`initial_x`参数指定一个位置来创建粒子。

In [ ]:
seed(6) 
run_pf1(N=5000, plot_particles=True, initial_x=(1,1, np.pi/4))

这个方法很好。如果您有任何办法粗略估计初始位置，应该尝试在初始位置附近创建粒子。不要*太*小心——如果您将所有点都生成在一个位置附近，粒子可能不够分散，无法捕捉系统中的非线性。这是一个相当线性的系统，因此我们可以通过分布中的较小方差来获得更好的样本。显然，这取决于您的问题。增加粒子数量始终是获得更好样本的好方法，但处理成本可能会比您愿意支付的成本更高。

## 重要性采样

我已经回避了一个困难，现在我们必须面对它。有一些概率分布描述了我们的机器人的位置和移动。我们希望从该分布中绘制粒子样本，并使用MC方法计算积分。

我们的困难在于在许多问题中我们不知道分布。例如，跟踪的对象可能移动方式与我们的状态模型预测的非常不同。我们如何从一个未知的概率分布中抽取样本？

有一个来自统计学的定理叫做 [*重要性抽样*](https://en.wikipedia.org/wiki/Importance_sampling)。令人惊讶的是，它给了我们一种从不同的已知概率分布中抽取样本并用它们来计算未知概率分布属性的方法。这是一个让我感到高兴的奇妙定理。

这个想法很简单，我们已经使用过它。我们从已知的概率分布中抽取样本，但是根据我们感兴趣的分布对样本进行*加权*。然后，我们可以通过计算样本的加权平均值和加权方差来计算诸如平均值和方差等属性。

对于机器人本地化问题，我们从我们预测步骤中计算出的概率分布中抽取样本。换句话说，我们推断“机器人在那里，它可能朝这个方向和速度移动，因此它可能在这里”。然而，机器人可能做了完全不同的事情。它可能从悬崖上掉下来或被迫击炮击中。在每种情况下，概率分布都是不正确的。看起来我们受阻了，但实际上并不是，因为我们可以使用重要性采样。我们从可能不正确的概率分布中抽取粒子，然后根据粒子与测量匹配的程度对其进行加权。这种加权是基于真实的概率分布的，因此根据理论，得到的平均值、方差等将是正确的！

那怎么可能是真的呢？我会给你一些数学知识；如果你不打算超越机器人定位问题，可以安全地跳过这部分。然而，其他粒子滤波问题需要不同的重要性抽样方法，了解一些数学知识会有所帮助。此外，文献和网络上的大部分内容都使用数学公式，而不是我相当不精确的“想象一下……”的表述。如果你想要理解文献，就需要了解以下方程式。

我们有一个概率分布$\pi(x)$，我们想要从中取样。然而，我们不知道$\pi(x)$是什么；相反，我们只知道另一个概率分布$q(x)$。在机器人定位的背景下，$\pi(x)$是机器人的概率分布，但我们不知道它，$q(x)$是我们的测量概率分布，我们知道它。

具有概率分布$\pi(x)$的函数$f(x)$的期望值是：

$$\mathbb{E}\big[f(x)\big] = \int f(x)\pi(x)\, dx$$

由于我们不知道 $\pi(x)$ ，所以我们无法计算这个积分。我们知道另一个分布 $q(x)$ ，所以我们可以将其添加到积分中而不改变值，如下所示

$$\mathbb{E}\big[f(x)\big] = \int f(x)\pi(x)\frac{q(x)}{q(x)}\, dx$$

现在我们重新排列和分组项

$$\mathbb{E}\big[f(x)\big] = \int f(x)q(x)\, \,  \cdot \,  \frac{\pi(x)}{q(x)}\, dx$$

我们已知 $q(x)$ ，因此我们可以使用 MC 积分计算 $\int f(x)q(x)$。这就给我们留下了 $\pi(x)/q(x)$ 。这是一个比率，我们将其定义为 *权重*。这给了我们

$$\mathbb{E}\big[f(x)\big] = \sum\limits_{i=1}^N f(x^i)q(x^i)w(x^i)$$

也许这看起来有点抽象。如果我们想要计算粒子的平均值，我们会计算

$$\mu = \frac{1}{N}\sum\limits_{i=1}^N x^iw^i$$

这是我在本章前面给你的公式。

需要满足权重与比率 $\pi(x)/q(x)$ 成比例的要求。实际上，我们通常不知道确切的值，因此在实践中，我们通过将权重除以 $\sum w(x^i)$ 进行归一化。

当您制定粒子滤波器算法时，您将不得不根据您的具体情况实施此步骤。对于机器人定位，用于 $q(x)$ 的最佳分布是滤波器的 `predict()` 步骤的粒子分布。让我们再次看一下代码：

In [ ]:
def update(particles, weights, z, R, landmarks):
    for i, landmark in enumerate(landmarks):
        dist = np.linalg.norm(particles[:, 0:2] - landmark, axis=1)
        weights *= scipy.stats.norm(dist, R).pdf(z[i])

    weights += 1.e-300      # 避免舍入为零
    weights /= sum(weights) # 归一化

在此处，我们通过贝叶斯计算 $\| \text{likelihood} \times \text{prior}\|$ 来计算权重。

当然，如果你能够从先验中计算后验概率分布，那么你应该这样做。如果不能，那么重要性采样为你提供了解决这个问题的方法。在实践中，计算后验概率是非常困难的。卡尔曼滤波器之所以成为一个惊人的成功，是因为它利用了高斯的属性找到了一个解析解。一旦我们放宽了卡尔曼滤波器所要求的条件（马尔可夫性质，高斯测量和过程），重要性采样和蒙特卡罗方法使问题变得易于处理。

## 重采样方法

重采样算法会影响滤波器的性能。例如，假设我们通过随机选择粒子来对粒子进行重采样。这将导致我们选择许多具有非常低权重的粒子，而得到的粒子集将是问题概率分布的一个糟糕的表示。

研究仍在继续，但是少数算法在各种情况下都表现良好。我们希望算法具有几个特性。它应该优先选择具有更高概率的粒子。它应该选择高概率粒子的代表性群体，以避免样本贫化。它应该包括足够的低概率粒子，以使滤波器有机会检测到强非线性行为。

FilterPy实现了几种流行的算法。FilterPy不知道你的粒子滤波器是如何实现的，因此无法生成新样本。相反，算法创建一个包含所选粒子索引的`numpy.array`。你的代码需要执行重采样步骤。例如，我在机器人中使用了这个：

In [ ]:
def resample_from_index(particles, weights, indexes):
    particles[:] = particles[indexes]
    weights.resize(len(particles))
    weights.fill(1.0 / len(weights))

### 多项式重采样

在开发机器人定位示例时，我使用了多项式重采样算法。这个想法很简单。计算归一化权重的累积和。这给你一个从0到1递增值的数组。这里有一个图，说明了这是如何将权重分隔开的。颜色是无意义的，只是为了让划分更容易看到。

In [ ]:
from kf_book.pf_internal import plot_cumsum
print('cumulative sume is', np.cumsum([.1, .2, .1, .6]))
plot_cumsum([.1, .2, .1, .6])

要选择一个权重，我们生成一个在0到1之间均匀选择的随机数，并使用二分法在累积和数组中找到其位置。大权重占据的空间比小权重多，所以它们更有可能被选中。

使用NumPy的[ufunc](http://docs.scipy.org/doc/numpy/reference/ufuncs.html)支持编写代码非常容易。Ufuncs将函数应用于数组的每个元素，返回结果数组。 `searchsorted`是NumPy的二分搜索算法。如果您提供一个搜索值数组，它将返回一个答案数组：每个搜索值的单个答案。

In [ ]:
def multinomal_resample(weights):
    cumulative_sum = np.cumsum(weights)
    cumulative_sum[-1] = 1.  # 避免舍入误差
    return np.searchsorted(cumulative_sum, random(len(weights)))    

这里是一个例子：

In [ ]:
from kf_book.pf_internal import plot_multinomial_resample
plot_multinomial_resample([.1, .2, .3, .4, .2, .3, .1])

这是一个$O(n \log(n))$的算法。虽然不算太糟糕，但是有$O(n)$的重新采样算法，相对于样本的均匀性有更好的性质。我展示这个算法是因为你可以将其他算法理解为这个算法的变体。有一种更快的实现多项式重新采样的方法，它使用了分布函数的反函数。如果你感兴趣，可以在互联网上搜索。

使用以下代码从FilterPy中导入函数：

In [ ]:
from filterpy.monte_carlo import multinomal_resample

### 残差重新采样

残差重采样不仅提高了多项式重采样的运行时间，而且确保了对粒子群的采样是均匀的。这相当巧妙：标准化权重乘以 *N*，然后使用每个权重的整数值来定义将取多少个该粒子的样本。例如，如果一个粒子的权重为0.0012，$N$ = 3000，则缩放后的权重为3.6，因此将取该粒子的3个样本。这确保了选择至少一次所有更高权重的粒子。运行时间为 $O(N)$，比多项式重采样更快。

然而，这并不能生成所有*N*个选择。为了选择剩余的，我们将*残差*取出：权重减去整数部分，这样就留下了数字的小数部分。然后，我们使用一种更简单的采样方案，如多项式，根据残差选择其余的粒子。在上面的例子中，缩放后的权重为3.6，因此残差将为0.6（3.6-int（3.6））。这个残差非常大，因此这个粒子很可能会被再次采样。这是合理的，因为残差越大，四舍五入误差就越大，因此整数步骤中的粒子相对被低估。

In [ ]:
def residual_resample(weights):
    N = len(weights)
    indexes = np.zeros(N, 'i')

    # 每个权重值取 int(N*w) 次
    num_copies = (N*np.asarray(weights)).astype(int)
    k = 0
    for i in range(N):
        for _ in range(num_copies[i]): # 重复 n 次
            indexes[k] = i
            k += 1

# 使用多项式重采样填充剩余部分。
    residual = w - num_copies     # 获取小数部分
    residual /= sum(residual)     # 归一化
    cumulative_sum = np.cumsum(residual)
    cumulative_sum[-1] = 1. # 确保总和为1
    indexes[k:N] = np.searchsorted(cumulative_sum, random(N-k))

    return indexes

你可能会想用切片`indexes[k:k + num_copies[i]] = i`替换内部循环，但是非常短的切片相对较慢，而且内部循环通常更快。

让我们看一个例子：

In [ ]:
from kf_book.pf_internal import plot_residual_resample
plot_residual_resample([.1, .2, .3, .4, .2, .3, .1])

你可以使用以下语句从FilterPy导入它

In [ ]:
    from filterpy.monte_carlo import residual_resample

### 分层重采样

这个方案旨在相对均匀地选择粒子。它的工作原理是将累积和分成$N$个相等的部分，然后从每个部分随机选择一个粒子。这保证了每个样本之间的距离在0到$\frac{2}{N}$之间。

下面的图表说明了这一点。彩色条形图显示了数组的累积和，黑线显示了$N$个等分。粒子（表示为黑色圆圈）随机放置在每个子分区中。

In [ ]:
from kf_book.pf_internal import plot_stratified_resample
plot_stratified_resample([.1, .2, .3, .4, .2, .3, .1])

执行分层采样的代码非常简单。

In [ ]:
def stratified_resample(weights):
    N = len(weights)
    # make N subdivisions, chose a random position within each one
    positions = (random(N) + range(N)) / N

In [ ]:
indexes = np.zeros(N, 'i')
    cumulative_sum = np.cumsum(weights)
    i, j = 0, 0
    while i < N:
        if positions[i] < cumulative_sum[j]:
            indexes[i] = j
            i += 1
        else:
            j += 1
    return indexes

从FilterPy中导入它

In [ ]:
from filterpy.monte_carlo import stratified_resample

### Systematic Resampling

我们将要讨论的最后一个算法是系统重采样。与分层重采样一样，空间被分成了$N$个部分。然后我们选择一个随机偏移来使用所有的部分，确保每个样本之间恰好相隔$\frac{1}{N}$。如下所示。

In [ ]:
from kf_book.pf_internal import plot_systematic_resample
plot_systematic_resample([.1, .2, .3, .4, .2, .3, .1])

在看过前面的例子之后，代码再简单不过了。

In [ ]:
def systematic_resample(weights):
    N = len(weights)

# 分成N个部分，选择位置
    # 使用一致的随机偏移量
    positions = (np.arange(N) + random()) / N

In [ ]:
indexes = np.zeros(N, 'i')
cumulative_sum = np.cumsum(weights)
i, j = 0, 0
while i < N:
    if positions[i] < cumulative_sum[j]:
        indexes[i] = j
        i += 1
    else:
        j += 1
return indexes
```

从FilterPy导入

In [ ]:
from filterpy.monte_carlo import systematic_resample
 ```

### 选择重采样算法

让我们一次性看四种算法，这样更容易比较。

```python
a = [.1, .2, .3, .4, .2, .3, .1]
np.random.seed(4)
plot_multinomial_resample(a)
plot_residual_resample(a)
plot_systematic_resample(a)
plot_stratified_resample(a)

多项式重采样的性能非常差。有一个非常大的权重根本没有被采样到。最大的权重只被重新采样了一次，而最小的权重被采样了两次。我阅读的大多数网络教程都使用多项式重采样，我不确定为什么。多项式重采样在文献或实际问题中很少使用。除非有非常好的理由，否则我建议不要使用它。

剩余重采样算法在其试图做的事情上表现出色：确保所有最大的权重都被多次重采样。它不会平均分配粒子之间的样本 - 许多相当大的权重根本没有被重新采样。

系统抽样和分层抽样都表现非常出色。系统抽样在确保我们从所有颗粒空间的部分进行采样的同时，确保更大的权重按比例被重新采样得更频繁。分层重采样不像系统重采样那么均匀，但它在确保更高权重得到重新采样方面要好一些。

关于这些算法的理论性能已经写了很多，可以自由阅读。在实践中，我将粒子滤波器应用于抵抗分析努力的问题，因此我对特定分析对这些问题的有效性有些怀疑。在实践中，分层和系统算法在多种问题上表现良好并且类似。我建议尝试其中一个，如果有效就坚持使用它。如果滤波器的性能很关键，请尝试两种方法，并查看是否有针对您具体问题的文献可以为您提供更好的指导。

## 总结

本章只是浅尝辄止，介绍了一个庞大的主题。我的目标不是教授这个领域，而是向您介绍用于滤波的实用贝叶斯蒙特卡罗技术。

粒子滤波器是一种*集合*滤波器。卡尔曼滤波器用高斯表示状态。贝叶斯定理将测量应用于高斯，而预测则使用状态空间方法。这些技术应用于高斯分布-概率分布。

相比之下，集合技术使用离散点集和相关概率来表示概率分布。测量应用于这些点，而不是高斯分布。同样，系统模型应用于这些点，而不是高斯分布。然后，我们计算所得到的点集的统计特性。

这些选择都有很多权衡。卡尔曼滤波器非常高效，在线性和高斯噪声假设成立时是最优估计器。如果问题是非线性的，则必须线性化问题。如果问题是多模态的（跟踪多个对象），则卡尔曼滤波器无法表示。卡尔曼滤波器要求您知道状态模型。如果您不知道系统的行为，则性能较差。

相比之下，粒子滤波器可以处理任意的非解析概率分布。如果粒子数足够大，则粒子的集合形成了分布的准确近似。即使存在严重的非线性，也能表现出色。重要性采样使我们能够计算概率，即使我们不知道潜在的概率分布。蒙特卡罗技术可以替换其他滤波器所需的解析积分。

这种能力是有代价的。最明显的代价是滤波器对计算机的高计算和内存负担。不太明显的是，它们是娇气的。你必须小心避免粒子退化和分歧。证明你的滤波器的正确性可能非常困难。如果你正在处理多模态分布，你需要进一步对粒子进行聚类以确定多个对象的路径。当对象彼此靠近时，这可能非常困难。

有许多不同类别的粒子滤波器；我只描述了天真的SIS算法，并紧随其后的是表现良好的SIR算法。有许多类别的滤波器，每个类别中都有许多示例滤波器。需要一本小书来描述它们。

当你阅读有关粒子滤波器的文献时，你会发现文中充斥着积分。我们使用积分对概率分布进行计算，因此使用积分给作者提供了一种强大而紧凑的符号表示。你必须意识到，当你将这些方程式转化为代码时，你将用粒子来表示分布，而积分将被替换为对粒子的求和。如果你记住本章的核心思想，这些材料不应该令人畏惧。

## 参考文献

[1] *重要性采样*，维基百科。
https://zh.wikipedia.org/wiki/%E9%87%8D%E8%A6%81%E6%80%A7%E9%87%87%E6%A8%A3